# 08. Supabase 발표용 거래 시나리오

발표 전용 가상 계좌 여섯 거래를 DB에 저장한다. 정상적인 소액 거래와 잔액 전액 거래의 모델 점수를 비교한다. 손으로 작성한 시연 시나리오이며 모델 정확도 평가에 사용하지 않는다. 고정 UUID를 사용하므로 재실행 시 기존 결과를 반환한다. 같은 step 이력 제외·중복 요청·과거 거래 조회를 확인한다.

In [1]:
import json
import sys
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

import pandas as pd
from fastapi.testclient import TestClient

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fds_agent.api.main import app  # noqa: E402

client = TestClient(app)
# 공개 합성 원본과 구분하기 위해 발표 전용 계좌 이름을 사용한다.
events = [
    (10, "PAYMENT", 100, 10000, 0, "demo_v1_alice", "demo_v1_shop"),
    (10, "PAYMENT", 200, 9900, 100, "demo_v1_alice", "demo_v1_shop"),
    (11, "TRANSFER", 1000, 9700, 5000, "demo_v1_alice", "demo_v1_bob"),
    (11, "PAYMENT", 50, 6000, 300, "demo_v1_bob", "demo_v1_shop"),
    (12, "TRANSFER", 8700, 8700, 0, "demo_v1_alice", "demo_v1_carol"),
    (13, "CASH_OUT", 8700, 8700, 0, "demo_v1_carol", "demo_v1_cash"),
]
results = []
for index, event in enumerate(events):
    step, kind, amount, origin_balance, dest_balance, origin, dest = event
    payload = dict(
        transaction_id=str(uuid5(NAMESPACE_URL, f"fds-demo-v1/{index}")),
        step=step,
        type=kind,
        amount=amount,
        oldbalanceOrg=origin_balance,
        oldbalanceDest=dest_balance,
        nameOrig=origin,
        nameDest=dest,
    )
    response = client.post("/v1/transactions/score", json=payload)
    assert response.status_code == 200, (index, response.status_code)
    result = response.json()
    repeated = client.post("/v1/transactions/score", json=payload)
    assert repeated.status_code == 200 and repeated.json()["replayed"]
    detail = client.get("/v1/transactions/" + payload["transaction_id"])
    assert detail.status_code == 200
    values = detail.json()
    results.append(
        dict(
            transaction_id=payload["transaction_id"],
            step=step,
            type=kind,
            amount=amount,
            origin=origin,
            destination=dest,
            risk_score=result["risk_score"],
            is_fraud=result["is_fraud"],
            send_prior_count=values["features"]["send_prior_count"],
            receive_prior_count=values["features"]["receive_prior_count"],
        )
    )
table = pd.DataFrame(results)
assert table.iloc[0].send_prior_count == 0
assert table.iloc[1].send_prior_count == 0
assert table.iloc[2].send_prior_count == 2
assert table.iloc[4].send_prior_count == 3
history = client.get("/v1/transactions", params={"account": "demo_v1_alice", "before_step": 12})
assert history.status_code == 200 and len(history.json()) == 3
assert all(row["step"] < 12 for row in history.json())
display(table)
out = ROOT / "artifacts/reports/demo_v1"
out.mkdir(parents=True, exist_ok=True)
table.to_csv(out / "transactions.csv", index=False)
(out / "summary.json").write_text(
    json.dumps(
        dict(
            rows=len(table),
            alerts=int(table.is_fraud.sum()),
            same_step_exclusion_verified=True,
            retries_verified=True,
            history_filter_verified=True,
            purpose="Hand-authored presentation scenarios; no accuracy claims",
        ),
        indent=2,
    )
)
print("Demo transactions remain in Supabase. Summary:", out)

/Users/yueun/Desktop/FDS-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


,transaction_id,step,type,amount,origin,destination,risk_score,is_fraud,send_prior_count,receive_prior_count
0,c95aed1b-0b51-5e69-a356-48cc2bee04f9,10,PAYMENT,100,demo_v1_alice,demo_v1_shop,0.000783,False,0,0
1,52e7231e-807e-54b1-bb5d-963e02dc6ad3,10,PAYMENT,200,demo_v1_alice,demo_v1_shop,0.000783,False,0,0
2,3f81027a-7fa2-544e-af2e-c49a07dd7aed,11,TRANSFER,1000,demo_v1_alice,demo_v1_bob,0.000783,False,2,0
3,6c8267f0-5faa-5d00-b08e-58e500f128b8,11,PAYMENT,50,demo_v1_bob,demo_v1_shop,0.000783,False,0,2
4,b677a82f-3ce6-5fdf-8309-6f06b1befee3,12,TRANSFER,8700,demo_v1_alice,demo_v1_carol,1.000000,True,3,0
5,684af5fb-fb45-5001-af51-1ee004446edf,13,CASH_OUT,8700,demo_v1_carol,demo_v1_cash,1.000000,True,0,0


Demo transactions remain in Supabase. Summary: /Users/yueun/Desktop/FDS-AI-Agent/artifacts/reports/demo_v1
